### Modelling differential pairs with ideal transmission lines

![Differential TLine Model](images/Differential_Transmission_Line_Model.png)

[Article](https://www.researchgate.net/publication/255648346_Validation_Methods_for_S-parameter_Measurement_Based_Models_of_Differential_Transmission_Lines)

This model allows making a differential [model](diff_tline_ltspice.zip) in LTSpice which considers the distributed coupling, but also emulates the different propagation delay between odd mode and even mode and consequently mode conversion.  </br>One can use whatever geometrical configuration (twisted pairs, microstrips, striplines, etc.), provided the knowledge of $Z_{even}$, $Z_{odd}$, $T_{e}$, $T_{o}$ </br>. 

This information can be retrieved from transmission line calculators (e.g. Kicad, QUCS, Saturn)
___
$V_{NEXT} =  V_1 \cdot k_b$

$k_b = \dfrac{1}{2}\dfrac{Z_{even}-Z_{odd}}{Z_{even}+Z_{odd}}$

$V_{FEXT} = V_1 \dfrac{Len}{RT} \cdot k_f $

$k_f = \dfrac{1}{2}\left( \dfrac{1}{v_{odd}} - \dfrac{1}{v_{even}} \right)$



### uSimmics / QUCS simulations
In this example the near-end cross-talk (NEXT) and the far-end (FEXT) voltage are simulated with uSimmics on a microstripline. </br>

![uSim_NEXT_FEXT](./images/uSimmics_circuit.PNG)

The parameters  $Z_{even}$, $Z_{odd}$, $T_{even}$, $T_{odd}$, can be extracted from the Transmission line calculator:

![Microstrip_props](images/Microstripline_properties.PNG)

### $V_{NEXT}$ and $V_{FEXT}$ on LTSpice



The properties: $Z_{odd}, \ Z_{even}, \ T_o, \ T_e$ can be passed to the LTSpice model. 
This model is built on the topology of a differential pair with single transmission line as described in the article.

![DP_LTSpice](images/Differential_Transmission_Line_ltspice_sub.png)

$T_o = \dfrac{Len}{c}\sqrt{\epsilon_{eff,odd}}$

$T_e = \dfrac{Len}{c}\sqrt{\epsilon_{eff,even}}$

![LTSpice_circuit](images/LTSpice_circuit.PNG)

Left - uSimmics results, right - LTSPICE

![LTSpice_NEXT_FEXT](images/uSimmics_LTSpice_FEXT_NEXT.PNG)

In [ ]:
def kb(Zo, Ze):
    return (Ze-Zo)/(Ze+Zo)/2

def kf(vo, ve):
    return (1/vo-1/ve)/2

def dT(eps_r, L):
    c = 3e8
    return L/c*((eps_r)**0.5)

### Example: coupled microstrip L=150 mm, Spacing = 2 mm, Width = 1.9 mm, h =1 mm, thickness =35 µm, er = 4.3 --> Zo = 47.21, Ze = 53.64, To = 0.87 ns, Te = 0.928 ns
### er_odd = 4.3, er_even = 3.8
Va = 200e-3
L= 150e-3
RT= 0.5e-9
Ze = 53.64
Zo = 47.21
er_o = 3.06
er_e = 3.45

To = dT(er_o, L)
Te = dT(er_e, L)
print(f"To = {To:.2e} s")
print(f"Te = {Te:.2e} s")
vo = L/To
ve = L/Te
kb_value = kb(Zo, Ze)
kf_value = kf(vo, ve)
# Because the generator has a series resistance of 50 Ohm, the voltage at the input of the line is Va/2
Vnext=Va*kb_value/2
print(f"Vnext = {Vnext:.2e} V")
Vfext = Va*kf_value*L/RT/2
print(f"Vfext = {Vfext:.2e} V")

### $V_{NEXT}$ and $V_{FEXT}$ vs coupling Length

In this example, on the LTSpice model $T_o$ and $T_e$ vary with the length. By introducing the following directives:

- *.param c=3e8 er_o=3.06 er_e=3.45*

- *.step param L list 25.4e-3 83.8e-3 127e-3 177.8e-3 254e-3*

And parametrizing the $T_o$ and $T_e$ in the differential transmission line component:

- Zo=47.21 Ze=53.64 To={sqrt(er_o)*L/c} Te={sqrt(er_e)*L/c}

uSimmics /QUCS and LTSpice results:

![NEXT_FEXT vs Length](images/uSimmics_LTSpice_FEXT_NEXT_vs_Length.PNG)

### $V_{NEXT}$ and $V_{FEXT}$ vs line spacing

In this example, on the LTSpice model $Z_o$, $Z_e$, $T_o$ and $T_e$ vary with the spacing between the lines. 

S = [1 mm, 2 mm, 3mm, 10 mm]

As usual the values of $Z_o$ and $Z_e$ are taken from the transmission lines calculator of uSimmics as well as $\epsilon_{eff,odd}$ and $\epsilon_{eff,even}$

By introducing the following directives:

- *.param L 150e-3*

- *.param c 3e8*

- *.step param idx list 1 2 3 4*

- *.param S = table(idx, 1,1e-3, 2,2e-3, 3,3e-3, 4,10e-3)*
 
- *.param Zo = table(idx, 1,43.85, 2,47.21, 3,48.47, 4,49.94)*
 
- *.param Ze = table(idx, 1,56.60, 2,53.64, 3,52.38 , 4, 50.91)*
 
- *.param er_e=table(idx, 1,3.5, 2,3.45, 3,3.4 , 4, 3.29)*
 
- *.param er_o=table(idx, 1,2.9, 2,3.06, 3,3.12 , 4, 3.25)*

And again parametrizing $Z_o$, $Z_e$, $T_o$ and $T_e$ in the differential transmission line component:

Zo={Zo} Ze={Ze} To={sqrt(er_o)*L/c} Te={sqrt(er_e)*L/c}

uSimmics /QUCS and LTSpice results:

![NEXT_FEXT vs Spacing](images/uSimmics_LTSpice_FEXT_NEXT_vs_Spacing.PNG)

### $V_{NEXT}$ and $V_{FEXT}$ vs rise time

In this example, on the LTSpice model $T_o$ and $T_e$ does not vary. By introducing the following directive:

- *.step param RT list 0.5n 1n 2n*

The component parameters are fixed:

- Zo=47.21 Ze=53.64 To=0.87n Te=0.928n

uSimmics /QUCS and LTSpice results:

![NEXT_FEXT vs Length](images/uSimmics_LTSpice_FEXT_NEXT_vs_RT.PNG)